# 02 — BPE 토크나이저: 학습 실험 · Tokenizer 클래스 · 인코딩

CS336 Assignment 1 · **Phase 1 후반부**

이 노트북에서 다루는 문제:

| 문제 | 배점 | 검증 방법 |
|---|---|---|
| `train_bpe_tinystories` | 2 | 실행 + 서술형 |
| `train_bpe_expts_owt` | 2 | 실행 + 서술형 |
| `tokenizer` | 15 | `pytest tests/test_tokenizer.py` |
| `tokenizer_experiments` | 4 | 서술형 + `.npy` 산출물 |

---

## 전체 파이프라인에서 지금 위치

```
[01] 원시 텍스트 ──(train_bpe)──> vocab + merges
                                      │
[02] ───────────────────────────────┐ │
      ├ 실제 데이터셋에 학습 ────────┼─┘   ← train_bpe_tinystories / _owt
      ├ vocab/merges 디스크 직렬화 ─┤
      ├ Tokenizer 클래스 (encode/decode) ← tokenizer (15pt)
      └ 데이터셋 전체 인코딩 ────────> train.npy / valid.npy (uint16)
                                                  │
[03~05] Transformer + 옵티마이저 <────────────────┘
[06~08] 학습 · ablation · 리더보드
```

> **왜 중요한가**: 이 노트북의 최종 산출물인 `.npy` 토큰 배열이 Phase 4 이후 모든 학습의 **유일한 입력**이다.
> 여기서 인코딩이 잘못되면 모델이 아무리 맞아도 loss가 내려가지 않고, 원인 파악에 며칠이 날아간다.
> 그래서 **tiktoken GPT-2를 정답 오라클로 두고** 한 단계씩 대조하며 진행한다.

각 섹션은 01과 동일하게 **배경 → 문제가 요구하는 것 → 왜 중요한가 → 코드/답안** 순서.

---
# 0. 셋업

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

# 노트북이 notebooks/ 안에 있다고 가정하고 repo 루트를 sys.path에 추가
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
ART_DIR = REPO_ROOT / "artifacts"          # vocab/merges 저장 위치
ART_DIR.mkdir(exist_ok=True)

print("REPO_ROOT :", REPO_ROOT)
print("DATA_DIR  :", DATA_DIR, "(exists:", DATA_DIR.exists(), ")")
print("ART_DIR   :", ART_DIR)

In [ ]:
# 01에서 만든 구현 불러오기 (없으면 안내만 출력)
try:
    from cs336_basics.tokenizer.train_bpe import train_bpe
    print("train_bpe import OK")
except Exception as e:
    print("train_bpe import 실패:", type(e).__name__, e)
    print("→ 01 노트북에서 cs336_basics/tokenizer/train_bpe.py 를 먼저 완성할 것")

try:
    import tiktoken
    print("tiktoken OK (정답 오라클로 사용)")
except Exception as e:
    print("tiktoken 없음:", e, "→ uv add tiktoken")

---
# 1. 데이터 다운로드

아직 데이터를 받지 않았다면 여기서 받는다. **Git Bash에서 실행** (노트북 아님).

```bash
cd /c/path/to/study-ai-ml/cs336-assignment1
mkdir -p data && cd data

curl -L -O https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz
curl -L -O https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_valid.txt.gz
curl -L -O https://huggingface.co/datasets/stanford-cs336/tinystories-sample/resolve/main/TinyStoriesV2-GPT4-train.txt
curl -L -O https://huggingface.co/datasets/stanford-cs336/tinystories-sample/resolve/main/TinyStoriesV2-GPT4-valid.txt

gunzip owt_train.txt.gz
gunzip owt_valid.txt.gz
cd ..
```

> **주의**: 정확한 URL은 과제 저장소 `README.md`의 다운로드 섹션이 최종 근거다.
> 링크가 바뀌었으면 그쪽을 따를 것. 대략 TinyStories ~2GB, OWT ~11GB.

**성공 판정**: 아래 셀에서 4개 파일이 모두 보이고 크기가 0이 아니면 OK.

In [ ]:
for name in [
    "TinyStoriesV2-GPT4-train.txt",
    "TinyStoriesV2-GPT4-valid.txt",
    "owt_train.txt",
    "owt_valid.txt",
]:
    p = DATA_DIR / name
    if p.exists():
        print(f"{name:36s} {p.stat().st_size / 1e9:8.3f} GB")
    else:
        print(f"{name:36s} [없음]")

In [ ]:
# 데이터가 실제로 어떻게 생겼는지 눈으로 확인 (문서 구분자 <|endoftext|> 확인이 목적)
p = DATA_DIR / "TinyStoriesV2-GPT4-valid.txt"
if p.exists():
    with open(p, encoding="utf-8") as f:
        head = f.read(1200)
    print(head)
    print("---")
    print("<|endoftext|> 포함:", "<|endoftext|>" in head)

---
# 2. vocab / merges 직렬화

## 배경
`train_bpe`가 돌려주는 것은 메모리 상의 `vocab: dict[int, bytes]`, `merges: list[tuple[bytes, bytes]]`다.
디스크에 저장해야 하는데 **JSON은 `bytes`를 직접 담지 못한다.** 임의의 바이트(0x00~0xFF) 중에는
UTF-8로 디코딩되지 않는 값이 많기 때문이다.

GPT-2는 이 문제를 **bytes → 안전한 유니코드 문자**로의 가역 매핑(`gpt2_bytes_to_unicode`)으로 푼다.
256개 바이트 각각을 "출력 가능하고 공백이 아닌" 유니코드 코드포인트 하나에 1:1 대응시킨다.
예: 공백 `0x20` → `'Ġ'`, 개행 `0x0A` → `'Ċ'`.

과제의 `tests/common.py`에 이 함수가 이미 있고, `tests/test_tokenizer.py`도 이 포맷으로 픽스처를 읽는다.

## 문제가 요구하는 것
`train_bpe_tinystories` / `train_bpe_expts_owt` 모두 "**serialize the resulting vocabulary and merges to disk
for further inspection**"을 명시한다. 즉 저장은 선택이 아니라 산출물이다.

## 왜 중요한가
- 10K/32K 학습은 수십 분~수 시간이 걸린다. 저장 안 하면 노트북 커널 재시작 = 처음부터 다시.
- 뒤 섹션의 `Tokenizer.from_files`가 바로 이 파일을 읽는다.
- 테스트 픽스처와 **같은 포맷**을 쓰면 내 토크나이저를 GPT-2 vocab으로도 돌려볼 수 있다.

In [ ]:
# 이제 cs336_basics/tokenizer/serialization.py 를 직접 import 해서 쓴다
from cs336_basics.tokenizer.serialization import (
    gpt2_bytes_to_unicode,
    load_vocab_merges,
    save_vocab_merges,
)

_b2u = gpt2_bytes_to_unicode()
_u2b = {v: k for k, v in _b2u.items()}
assert len(_b2u) == 256 and all(_u2b[_b2u[b]] == b for b in range(256))
print("gpt2_bytes_to_unicode 라운드트립 OK")
print(f"  0x20(space) -> {_b2u[0x20]!r}   0x0A(\\n) -> {_b2u[0x0A]!r}   0xFF -> {_b2u[0xFF]!r}")
print()
print("모듈 자체 자가검증:  uv run python -m cs336_basics.tokenizer.serialization")

In [ ]:
# (이 셀은 더 이상 필요 없음 — serialization.py 가 repo에 있음)
print("OK")

---
# 3. Problem (`train_bpe_tinystories`) — 2pt

## 문제가 요구하는 것
> (a) TinyStories에 vocab_size **10,000**으로 byte-level BPE 학습. `<|endoftext|>` 특수 토큰 포함.
> vocab/merges를 디스크에 직렬화. **학습 시간과 메모리는 얼마였나? vocab에서 가장 긴 토큰은? 말이 되는가?**
> 자원 제한: **≤ 30분 (GPU 없이), ≤ 30GB RAM**
>
> (b) **프로파일링. 학습 과정에서 어느 부분이 가장 오래 걸리는가?**

핸드아웃 힌트: multiprocessing pre-tokenization + `<|endoftext|>` 특수 처리로 **2분 이내**도 가능.

## 왜 중요한가
- 30분/30GB는 넉넉해 보이지만, 01에서 만든 증분 업데이트가 없으면 **몇 시간**이 걸린다.
  즉 이 문제는 사실상 01 최적화의 실전 검증이다.
- 프로파일링 결과(보통 pre-tokenization이 지배적)는 뒤에 OWT(5배 이상)로 갈 때
  어디를 병렬화할지 결정하는 근거가 된다.
- "가장 긴 토큰이 말이 되는가"는 데이터셋 성격을 드러낸다 — 이후 TinyStories vs OWT loss 차이
  (Phase 7 `main_experiment`)를 설명할 때 그대로 인용하게 된다.

## Windows 주의사항 ⚠️
`multiprocessing`은 Windows에서 **spawn** 방식이라 자식 프로세스가 모듈을 다시 import한다.
- 노트북 셀에 정의한 함수는 워커에서 pickle 되지 않아 **실패한다.**
- 반드시 **`.py` 파일에 정의된 top-level 함수**를 워커에 넘기고,
- 실행 스크립트에는 **`if __name__ == "__main__":` 가드**를 넣을 것.

그래서 학습은 노트북 셀이 아니라 **스크립트로 실행**한다.

In [ ]:
# scripts/train_tokenizer.py 가 준비되어 있는지 확인
script_path = REPO_ROOT / "scripts" / "train_tokenizer.py"
print("존재:", script_path.exists(), "|", script_path)
if script_path.exists():
    print()
    print(script_path.read_text(encoding="utf-8")[:1200], "...")
else:
    print("→ scripts/train_tokenizer.py 를 repo에 배치할 것")

### 실행 (Git Bash)

```bash
cd /c/path/to/study-ai-ml/cs336-assignment1

uv run python scripts/train_tokenizer.py \
  --input data/TinyStoriesV2-GPT4-train.txt \
  --vocab-size 10000 \
  --special-tokens '<|endoftext|>' \
  --out-prefix artifacts/tinystories_10k \
  --num-processes 8
```

**기대 출력**: `elapsed_sec`, `peak_mem_gb`, `longest_token_repr`가 담긴 JSON.
**성공 판정**: `artifacts/tinystories_10k-vocab.json` / `-merges.txt` / `-stats.json` 3개 생성 + `elapsed_sec < 1800`.
**실패 시**: 30분을 넘기면 01의 증분 업데이트(`pair_counts` + `pair_to_words`)가 실제로 동작하는지 먼저 의심할 것.

> `--num-processes`는 물리 코어 수 정도로. 너무 크게 잡으면 chunk가 잘게 쪼개져 오버헤드만 커진다.

### 프로파일링 (문제 (b))

```bash
uv run python -m cProfile -o artifacts/ts_profile.out scripts/train_tokenizer.py \
  --input data/TinyStoriesV2-GPT4-valid.txt \
  --vocab-size 10000 \
  --out-prefix artifacts/_profile_scratch \
  --num-processes 1
```

전체 train 파일 대신 **valid(작은 파일) + 단일 프로세스**로 프로파일링하는 이유:
멀티프로세싱은 자식 프로세스의 시간이 `cProfile`에 안 잡혀서 결과가 왜곡된다.
병목의 *상대적 비중*만 보면 되므로 작은 데이터로 충분하다.

In [ ]:
import pstats

prof = ART_DIR / "ts_profile.out"
if prof.exists():
    st = pstats.Stats(str(prof))
    st.strip_dirs().sort_stats("cumulative").print_stats(15)
else:
    print("프로파일 결과 없음 — 위 cProfile 명령을 먼저 실행할 것")

In [ ]:
# 학습 결과 확인: 가장 긴 토큰 / 토큰 길이 분포
from collections import Counter

def inspect_vocab(prefix: Path, top_k: int = 10):
    vpath, mpath = f"{prefix}-vocab.json", f"{prefix}-merges.txt"
    if not Path(vpath).exists():
        print(f"[없음] {vpath} — 먼저 학습을 돌릴 것")
        return None, None
    vocab, merges = load_vocab_merges(vpath, mpath)
    print(f"=== {prefix.name} ===")
    print("vocab size :", len(vocab))
    print("merges     :", len(merges))

    by_len = sorted(vocab.values(), key=len, reverse=True)
    print(f"\n가장 긴 토큰 {top_k}개:")
    for t in by_len[:top_k]:
        print(f"  len={len(t):3d}  {t!r}")

    dist = Counter(len(t) for t in vocab.values())
    print("\n토큰 길이 분포:", dict(sorted(dist.items())))

    if Path(f"{prefix}-stats.json").exists():
        print("\nstats:", json.load(open(f"{prefix}-stats.json", encoding="utf-8")))
    return vocab, merges


ts_vocab, ts_merges = inspect_vocab(ART_DIR / "tinystories_10k")

### 📝 서술형 답안 초안 — `train_bpe_tinystories`

> 실행 후 실제 숫자로 채울 것. 아래는 뼈대.

**(a)** TinyStories(2.1GB)에 vocab 10,000 학습에 `___초`(약 `__`분), peak 메모리 `__GB`가 걸렸다
(N 프로세스 pre-tokenization + 증분 pair count 갱신). 가장 긴 토큰은 `___`로 길이 `__` 바이트인데,
TinyStories가 소수의 정형화된 문장 패턴을 반복하는 합성 아동 동화 데이터셋이라
`b' accidentally'` 같은 **긴 단어/상투구가 통째로 하나의 토큰이 되는 것은 자연스럽다.**

**(b)** 프로파일 상 `___`(보통 pre-tokenization의 regex `finditer`)가 전체의 `__%`로 가장 오래 걸렸고,
merge 루프는 `__%`였다. pre-tokenization은 문서 단위로 독립적이라 병렬화가 가능한 반면,
merge 루프는 순차 의존성이 있어 병렬화가 안 된다 — 그래서 최적화 여력이 남은 쪽은 **증분 갱신의 상수 계수**다.

---
# 4. Problem (`train_bpe_expts_owt`) — 2pt

## 문제가 요구하는 것
> (a) OpenWebText에 vocab_size **32,000**으로 학습, 직렬화. **가장 긴 토큰은? 말이 되는가?**
> 자원 제한: ≤ 12시간 (GPU 없이), ≤ 100GB RAM
> (b) **TinyStories 토크나이저와 비교·대조.**

## 왜 중요한가
- OWT는 실제 웹 텍스트라 **노이즈**(HTML 잔재, 반복 구분선, 이상한 유니코드)가 있다.
  BPE는 "자주 나오는 것"을 무조건 토큰으로 만들기 때문에 이 노이즈가 vocab을 오염시킨다.
  → 가장 긴 토큰이 `b'----------------'` 같은 게 나오는 이유.
- 이 대비가 Phase 7 `main_experiment`("같은 compute인데 왜 OWT loss가 더 높은가")의 직접적 근거가 된다.

## ⚠️ 4GB VRAM / 개인 PC 현실 체크
OWT 32K는 GPU를 안 쓰지만 **RAM과 시간**을 많이 먹는다. 100GB RAM을 가정한 문제다.
개인 Windows 머신이면:
1. 먼저 `owt_valid.txt`(작음)로 파이프라인이 도는지 확인,
2. 그다음 `owt_train.txt` 전체를 돌리되 **메모리 모니터링**을 켜둘 것,
3. 메모리가 터지면 `train_bpe` 내부에서 pre-token 카운트를 `Counter`로 합칠 때
   chunk별 결과를 **즉시 합산하고 버리는지**(전부 리스트에 모아두지 않는지) 확인.

### 실행 (Git Bash)

```bash
# 1단계: 작은 파일로 파이프라인 검증 (몇 분)
uv run python scripts/train_tokenizer.py \
  --input data/owt_valid.txt \
  --vocab-size 32000 \
  --out-prefix artifacts/owt_valid_32k_smoke \
  --num-processes 8

# 2단계: 본 학습 (수 시간)
uv run python scripts/train_tokenizer.py \
  --input data/owt_train.txt \
  --vocab-size 32000 \
  --out-prefix artifacts/owt_32k \
  --num-processes 8
```

**성공 판정**: `artifacts/owt_32k-stats.json`이 생기고 `elapsed_sec < 43200`.
**중간 확인**: 1단계 smoke run의 `longest_token_repr`가 이미 TinyStories와 확연히 달라야 정상이다.

In [ ]:
owt_vocab, owt_merges = inspect_vocab(ART_DIR / "owt_32k")

In [ ]:
# TinyStories vs OWT 정량 비교
def compare_vocabs(v1, name1, v2, name2):
    if v1 is None or v2 is None:
        print("두 vocab이 모두 있어야 비교 가능")
        return
    s1, s2 = set(v1.values()), set(v2.values())
    avg = lambda s: sum(len(t) for t in s) / len(s)          # noqa: E731
    print(f"{'':22s}{name1:>14s}{name2:>14s}")
    print(f"{'vocab size':22s}{len(s1):>14d}{len(s2):>14d}")
    print(f"{'평균 토큰 길이(byte)':22s}{avg(s1):>14.2f}{avg(s2):>14.2f}")
    print(f"{'최장 토큰 길이':22s}{max(len(t) for t in s1):>14d}{max(len(t) for t in s2):>14d}")
    print(f"\n공통 토큰: {len(s1 & s2)}  |  {name1} 전용: {len(s1 - s2)}  |  {name2} 전용: {len(s2 - s1)}")

    print(f"\n{name2}에만 있는 토큰 샘플 (긴 것 위주):")
    for t in sorted(s2 - s1, key=len, reverse=True)[:15]:
        print("   ", repr(t))
    print(f"\n{name1}에만 있는 토큰 샘플:")
    for t in sorted(s1 - s2, key=len, reverse=True)[:15]:
        print("   ", repr(t))


compare_vocabs(ts_vocab, "TinyStories", owt_vocab, "OWT")

### 📝 서술형 답안 초안 — `train_bpe_expts_owt`

**(a)** OWT 32K 학습에 `___`시간, peak `__GB`. 가장 긴 토큰은 `___`(길이 `__`).
TinyStories의 최장 토큰이 실제 영단어였던 것과 달리, OWT 쪽은 `___`처럼
**웹 문서의 서식 잔재/반복 구분선**이 올라오는 경향이 있다. 이는 BPE가 의미가 아니라
순수 빈도만 보기 때문이며, 정제되지 않은 웹 크롤 데이터에서는 "말이 되는" 결과다
(다만 **언어 모델 입장에서 유용한 토큰은 아니다**).

**(b)** TinyStories(10K)는 어휘가 제한된 합성 아동 동화라 vocab이 짧고 흔한 영단어에 집중되는 반면,
OWT(32K)는 도메인이 넓어 고유명사·URL 조각·코드/마크업 파편·비영어 문자열까지 포함한다.
공통 토큰은 `___`개에 불과해, **TinyStories 토크나이저로 OWT를 인코딩하면 압축률이 크게 나빠진다**
(다음 섹션 실험 (b)에서 정량 확인).

---
# 5. Problem (`tokenizer`) — 15pt

## 배경: 인코딩 알고리즘

학습(`train_bpe`)이 "어떤 pair를 합칠지 **찾는**" 과정이었다면,
인코딩은 "이미 정해진 merge 목록을 **순서대로 적용하는**" 과정이다.

```
text
 ├─ Step 0. 특수 토큰으로 분리 (특수 토큰은 merge 대상이 아님)
 ├─ Step 1. 각 세그먼트를 GPT-2 regex로 pre-tokenize
 ├─ Step 2. 각 pre-token을 UTF-8 바이트 리스트로 (b'the' -> [b't', b'h', b'e'])
 ├─ Step 3. merges를 생성 순서대로 적용 (pre-token 경계를 넘지 않음)
 └─ Step 4. 최종 바이트 조각 -> vocab의 int ID
```

### 핸드아웃 예제로 감 잡기
vocab `{0:b' ',1:b'a',2:b'c',3:b'e',4:b'h',5:b't',6:b'th',7:b' c',8:b' a',9:b'the',10:b' at'}`,
merges `[(t,h), (' ',c), (' ',a), (th,e), (' a',t)]`, 입력 `'the cat ate'` → `[9, 7, 1, 5, 10, 3]`

## 문제가 요구하는 것
`Tokenizer` 클래스, 인터페이스 4개:

| 메서드 | 시그니처 | 핵심 요구 |
|---|---|---|
| `__init__` | `(vocab, merges, special_tokens=None)` | 특수 토큰이 vocab에 없으면 **추가** |
| `from_files` | `@classmethod (vocab_filepath, merges_filepath, special_tokens=None)` | 직렬화 포맷에서 복원 |
| `encode` | `(text: str) -> list[int]` | |
| `encode_iterable` | `(iterable: Iterable[str]) -> Iterator[int]` | **제너레이터, 메모리 상수** |
| `decode` | `(ids: list[int]) -> str` | 깨진 UTF-8은 U+FFFD |

## 왜 중요한가
15점짜리이자, 여기서 만든 게 **틀려도 조용히 틀린다**는 게 무섭다.
BPE 인코딩이 미묘하게 다르면 모델은 그냥 조금 더 나쁜 loss를 낼 뿐, 에러를 안 낸다.
그래서 `tests/test_tokenizer.py`가 **tiktoken GPT-2와 ID 단위로 완전 일치**를 요구한다.

## 함정 4가지 (테스트가 실제로 저격하는 것들)

### ① 특수 토큰은 **긴 것부터** 매칭해야 한다
`test_overlapping_special_tokens`는 `special_tokens=["<|endoftext|>", "<|endoftext|><|endoftext|>"]`로
`"... <|endoftext|><|endoftext|> ..."`를 인코딩했을 때 **더블이 하나의 토큰으로 보존**되기를 요구한다.
정규식 대안(`|`)은 **왼쪽 우선**이라, 정렬하지 않으면 짧은 쪽이 먼저 먹는다.

### ② `re.split`에 **캡처 그룹**을 써야 특수 토큰이 살아남는다
`re.split(r"(...)", text)`처럼 괄호로 감싸면 구분자도 결과 리스트에 포함된다.
안 감싸면 특수 토큰이 그냥 사라져서 `decode(encode(x)) != x`가 된다.

### ③ `decode`는 **바이트를 전부 이어붙인 뒤 한 번에** 디코딩
ID 하나씩 `.decode()`하면 여러 바이트짜리 문자(한글, 이모지)가 조각나서 깨진다.
→ 01의 `unicode2 (b)`에서 본 그 버그와 같은 구조. `errors="replace"`는 마지막에 한 번만.

### ④ `encode_iterable`은 **1MB 메모리 제한**을 받는다
`test_encode_iterable_memory_usage`가 5MB 파일을 1MB 제한으로 돌린다.
`"".join(iterable)`이나 `f.read()`를 하는 순간 **탈락**.
반대로 `test_encode_memory_usage`는 `encode`가 초과하는 게 **정상**이라 `xfail`로 잡혀 있다.

In [ ]:
# 함정 ①② 검증: 특수 토큰 분리를 실제로 돌려보기
import re

def split_on_specials(text: str, special_tokens: list[str]):
    if not special_tokens:
        return [text]
    ordered = sorted(special_tokens, key=len, reverse=True)   # ← 긴 것 우선
    pattern = "(" + "|".join(re.escape(s) for s in ordered) + ")"
    return re.split(pattern, text)


text = "Hello, how <|endoftext|><|endoftext|> are you?<|endoftext|>"
specials = ["<|endoftext|>", "<|endoftext|><|endoftext|>"]

print("정렬 O:", split_on_specials(text, specials))

bad = "(" + "|".join(re.escape(s) for s in specials) + ")"    # 정렬 안 함
print("정렬 X:", re.split(bad, text))

assert "".join(split_on_specials(text, specials)) == text, "무손실 분리 실패"
print("\n→ 정렬 O 에서만 더블 endoftext가 한 덩어리로 유지된다")

In [ ]:
# 함정 ③ 검증: decode를 ID 하나씩 하면 깨진다
pieces = [b"\xed\x95", b"\x9c"]          # '한' (U+D55C) 의 UTF-8 3바이트를 쪼갠 것

wrong = "".join(p.decode("utf-8", errors="replace") for p in pieces)
right = b"".join(pieces).decode("utf-8", errors="replace")
print("조각별 디코딩:", repr(wrong))
print("이어붙인 뒤   :", repr(right))
assert right == "한"

## 구현 스켈레톤 — `cs336_basics/tokenizer/tokenizer.py`

> **`_bpe` / `encode_iterable` / `decode` 의 TODO는 직접 채울 것.** 여기가 이 문제의 학습 포인트다.
> 나머지(특수 토큰 처리, 캐시, 직렬화 연결)는 보일러플레이트라 그대로 써도 된다.

```python
from __future__ import annotations

import re
from collections.abc import Iterable, Iterator

import regex  # GPT-2 패턴은 \p{L} 등을 쓰므로 표준 re가 아니라 regex 패키지 필요

from cs336_basics.tokenizer.serialization import load_vocab_merges

PAT = r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"


class Tokenizer:
    def __init__(self, vocab, merges, special_tokens=None):
        self.vocab = dict(vocab)                    # int -> bytes
        self.merges = list(merges)
        self.special_tokens = list(special_tokens or [])

        # 특수 토큰이 vocab에 없으면 추가
        existing = set(self.vocab.values())
        for st in self.special_tokens:
            b = st.encode("utf-8")
            if b not in existing:
                self.vocab[len(self.vocab)] = b
                existing.add(b)

        self.byte_to_id = {b: i for i, b in self.vocab.items()}
        self.ranks = {pair: i for i, pair in enumerate(self.merges)}

        # 특수 토큰 분리 패턴 — 긴 것 우선!
        self._special_pat = None
        if self.special_tokens:
            ordered = sorted(self.special_tokens, key=len, reverse=True)
            self._special_pat = "(" + "|".join(re.escape(s) for s in ordered) + ")"

        self._pat = regex.compile(PAT)
        self._cache: dict[bytes, list[int]] = {}
        self._cache_max = 100_000

    @classmethod
    def from_files(cls, vocab_filepath, merges_filepath, special_tokens=None):
        vocab, merges = load_vocab_merges(vocab_filepath, merges_filepath)
        return cls(vocab, merges, special_tokens)

    # ---------------- 여기가 핵심 구현부 ---------------- #
    def _bpe(self, token: bytes) -> list[int]:
        # 하나의 pre-token(bytes)에 merge를 적용해 ID 리스트로 변환
        cached = self._cache.get(token)
        if cached is not None:
            return cached

        parts = [bytes([b]) for b in token]

        # TODO(1): rank 방식으로 merge 적용
        #   while True:
        #     a) 인접쌍 (parts[i], parts[i+1]) 중 self.ranks 값이 최소인 위치 i 를 찾는다
        #     b) 그런 i 가 없으면 break
        #     c) parts[i:i+2] = [parts[i] + parts[i+1]]
        raise NotImplementedError

        ids = [self.byte_to_id[p] for p in parts]
        if len(self._cache) < self._cache_max:
            self._cache[token] = ids
        return ids

    def encode(self, text: str) -> list[int]:
        ids: list[int] = []
        segments = re.split(self._special_pat, text) if self._special_pat else [text]
        special_set = set(self.special_tokens)

        for seg in segments:
            if not seg:
                continue
            if seg in special_set:
                ids.append(self.byte_to_id[seg.encode("utf-8")])
                continue
            for m in self._pat.finditer(seg):
                ids.extend(self._bpe(m.group().encode("utf-8")))
        return ids

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        # 메모리 상수. 절대 전체를 join / read 하지 말 것
        # TODO(2): 방식 B(줄 단위 그대로 encode) 또는
        #          방식 C(버퍼에 모으되 buf.rfind("\\n")+1 위치, 즉 개행 뒤에서만 flush)
        #
        # 절대 하지 말 것: 고정 N글자 홀드백 / 임의 크기 청크.
        # 단어 중간이 잘려 encode()와 다른 ID가 나오는데 라운드트립은 통과 -> 조용히 망가진다.
        raise NotImplementedError

    def decode(self, ids: list[int]) -> str:
        # TODO(3): 모든 ID 의 bytes 를 이어붙인 뒤 한 번에 decode(errors="replace")
        #   vocab 에 없는 ID 는 U+FFFD 로 대체
        raise NotImplementedError
```

### merge 적용, 두 가지 방식

**(A) 순진한 방식** — merges 를 순서대로 훑으며 매번 전체 스캔.
정확하지만 pre-token 하나당 `O(len(merges) x len)`. vocab 32K 에서는 너무 느리다.

**(B) rank 방식 (권장)** — merge 를 dict 로 뒤집어 "지금 적용 가능한 merge 중 가장 이른 것"만 찾는다.
pre-token 길이가 대부분 15바이트 미만이므로 훨씬 빠르다.

**두 방식은 반드시 같은 결과를 내야 한다** — 아래에 대조 셀을 둔다.

### ⚠️ `encode_iterable` 청크 경계 (실측 결과)

| 방식 | `encode()`와 일치? |
|---|---|
| 임의 크기 청크를 그냥 encode | ❌ 8케이스 중 5개 불일치 |
| 고정 N글자 홀드백 | ❌ 단어 중간이 잘림 |
| **줄 단위 (파일 핸들 순회)** | ✅ 전부 일치 |
| **버퍼 + 개행 뒤에서만 flush** | ✅ 전부 일치 |

특히 무서운 건 **틀린 방식도 `decode(encode(x)) == x`는 통과한다**는 점이다.
라운드트립만 확인하고 넘어가면 조용히 다른 토큰으로 학습하게 된다.
반드시 `encode_iterable`의 결과를 `encode` 전체 결과와 **ID 단위로** 대조할 것.

### 캐시
같은 pre-token(`' the'` 등)이 수백만 번 반복된다. `bytes -> list[int]` 캐싱이 큰 이득.
단 `encode_iterable` 의 1MB 제한 때문에 **캐시 크기 상한**을 둘 것.

### `tests/adapters.py` 연결

```python
def get_tokenizer(vocab, merges, special_tokens=None):
    from cs336_basics.tokenizer.tokenizer import Tokenizer
    return Tokenizer(vocab, merges, special_tokens)
```

> **glue만.** 로직을 여기 넣지 말 것.

In [ ]:
# 구현 후 자가 검증 ① — 핸드아웃 bpe_encoding 예제
try:
    from cs336_basics.tokenizer.tokenizer import Tokenizer

    toy_vocab = {0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't',
                 6: b'th', 7: b' c', 8: b' a', 9: b'the', 10: b' at'}
    toy_merges = [(b't', b'h'), (b' ', b'c'), (b' ', b'a'), (b'th', b'e'), (b' a', b't')]

    tok = Tokenizer(toy_vocab, toy_merges)
    got = tok.encode("the cat ate")
    print("got     :", got)
    print("expected: [9, 7, 1, 5, 10, 3]")
    assert got == [9, 7, 1, 5, 10, 3], "핸드아웃 예제 불일치"
    assert tok.decode(got) == "the cat ate"
    print("✅ 핸드아웃 예제 통과")
except NotImplementedError:
    print("아직 미구현 — tokenizer.py의 TODO를 채울 것")
except Exception as e:
    print("실패:", type(e).__name__, e)

In [ ]:
# 구현 후 자가 검증 ② — 순진한 방식 vs rank 방식 대조 (내 구현이 rank 방식일 때 정합성 확인)
def bpe_naive(token: bytes, merges) -> list[bytes]:
    parts = [bytes([b]) for b in token]
    for a, b in merges:
        i = 0
        while i < len(parts) - 1:
            if parts[i] == a and parts[i + 1] == b:
                parts[i:i + 2] = [a + b]
            else:
                i += 1
    return parts


try:
    import random
    random.seed(0)
    if ts_vocab is not None:
        t = Tokenizer(ts_vocab, ts_merges, ["<|endoftext|>"])
        samples = ["the", " little", " because", " Once upon a time", "Tim", " happily", "!!!"]
        ok = True
        for s in samples:
            for m in t._pat.finditer(s):
                w = m.group().encode()
                naive_ids = [t.byte_to_id[p] for p in bpe_naive(w, ts_merges)]
                mine = t._bpe(w)
                if naive_ids != mine:
                    ok = False
                    print("불일치:", w, naive_ids, mine)
        print("✅ 두 방식 일치" if ok else "❌ 불일치 발생")
    else:
        print("TinyStories vocab이 없어 생략")
except NameError:
    print("Tokenizer 미구현 — 생략")

In [ ]:
# 구현 후 자가 검증 ③ — tiktoken GPT-2 정답 대조 (테스트와 동일한 조건)
try:
    import tiktoken
    from tests.test_tokenizer import get_tokenizer_from_vocab_merges_path
    from tests.common import FIXTURES_PATH

    ref = tiktoken.get_encoding("gpt2")
    mine = get_tokenizer_from_vocab_merges_path(
        FIXTURES_PATH / "gpt2_vocab.json",
        FIXTURES_PATH / "gpt2_merges.txt",
        special_tokens=["<|endoftext|>"],
    )
    cases = [
        "",
        "s",
        "🙃",
        "Hello, how are you?",
        "Héllò hôw are ü? 🙃",
        "Héllò hôw <|endoftext|><|endoftext|> are ü? 🙃<|endoftext|>",
        "   leading and trailing   ",
        "line1\n\nline3\n",
    ]
    for c in cases:
        a = mine.encode(c)
        b = ref.encode(c, allowed_special={"<|endoftext|>"})
        status = "OK " if a == b else "MISMATCH"
        print(f"{status} {c!r:60s} len={len(a)}")
        assert mine.decode(a) == c, f"라운드트립 실패: {c!r}"
    print("\n✅ 전부 일치")
except Exception as e:
    print("실행 불가:", type(e).__name__, e)

### pytest로 최종 확인

```bash
# 하나씩
uv run pytest tests/test_tokenizer.py -k roundtrip -x -q
uv run pytest tests/test_tokenizer.py -k tiktoken -x -q
uv run pytest tests/test_tokenizer.py -k special -x -q

# 전체
uv run pytest tests/test_tokenizer.py -q
```

**기대 출력**: Windows에서는 메모리 제한 테스트 2개가 `skipped`(rlimit이 Linux 전용)로 나온다.
그 외 전부 `passed`면 15점 완료.

**실패 유형별 진단**
| 증상 | 원인 |
|---|---|
| `test_overlapping_special_tokens` 실패 | 특수 토큰을 길이 내림차순으로 정렬 안 함 |
| `decode(encode(x)) != x` | `re.split`에 캡처 그룹 누락, 또는 decode를 ID별로 수행 |
| `*_matches_tiktoken` 실패 | merge 적용 순서 오류 (rank 최소가 아닌 왼쪽 우선으로 합침) |
| 인코딩이 아주 느림 | pre-token 캐시 없음, 또는 순진한 merge 방식 사용 |

---
# 6. Problem (`tokenizer_experiments`) — 4pt

네 개 소문제. (d)의 `.npy` 산출물이 Phase 4 이후의 입력이다.

> (a) TinyStories/OWT에서 각 10개 문서 샘플 → 각 토크나이저로 인코딩 → **압축률(bytes/token)**?
> (b) OWT 샘플을 **TinyStories 토크나이저**로 인코딩하면? 압축률 비교/정성 서술
> (c) **throughput(bytes/second)** 추정 → Pile(825GB) 인코딩에 얼마나 걸리나?
> (d) 학습/검증 데이터셋 전체 인코딩 → **uint16 NumPy 배열**로 저장. **왜 uint16이 적절한가?**

In [ ]:
import random

def sample_documents(path: Path, n: int = 10, seed: int = 42, max_scan_docs: int = 20000):
    """<|endoftext|> 구분자 기준으로 문서를 읽어 n개 무작위 샘플. 파일 전체를 메모리에 올리지 않는다."""
    rng = random.Random(seed)
    docs, buf = [], []
    reservoir = []
    with open(path, encoding="utf-8", errors="ignore") as f:
        count = 0
        for line in f:
            if "<|endoftext|>" in line:
                head, _, tail = line.partition("<|endoftext|>")
                buf.append(head)
                doc = "".join(buf).strip()
                if doc:
                    count += 1
                    if len(reservoir) < n:                  # reservoir sampling
                        reservoir.append(doc)
                    elif rng.random() < n / count:
                        reservoir[rng.randrange(n)] = doc
                buf = [tail]
                if count >= max_scan_docs:
                    break
            else:
                buf.append(line)
    return reservoir


ts_docs = sample_documents(DATA_DIR / "TinyStoriesV2-GPT4-valid.txt") if (DATA_DIR / "TinyStoriesV2-GPT4-valid.txt").exists() else []
owt_docs = sample_documents(DATA_DIR / "owt_valid.txt") if (DATA_DIR / "owt_valid.txt").exists() else []

print(f"TinyStories 샘플 {len(ts_docs)}개, 평균 {sum(len(d) for d in ts_docs)/max(len(ts_docs),1):.0f}자")
print(f"OWT 샘플 {len(owt_docs)}개, 평균 {sum(len(d) for d in owt_docs)/max(len(owt_docs),1):.0f}자")
if ts_docs:
    print("\n--- TinyStories 샘플 ---\n", ts_docs[0][:300])
if owt_docs:
    print("\n--- OWT 샘플 ---\n", owt_docs[0][:300])

In [ ]:
# (a) + (b) 압축률
def compression_ratio(tokenizer, docs) -> dict:
    total_bytes = sum(len(d.encode("utf-8")) for d in docs)
    t0 = time.perf_counter()
    total_tokens = sum(len(tokenizer.encode(d)) for d in docs)
    elapsed = time.perf_counter() - t0
    return {
        "bytes": total_bytes,
        "tokens": total_tokens,
        "bytes_per_token": total_bytes / total_tokens if total_tokens else float("nan"),
        "throughput_bps": total_bytes / elapsed if elapsed else float("nan"),
        "elapsed_sec": elapsed,
    }


try:
    from cs336_basics.tokenizer.tokenizer import Tokenizer

    ts_tok = Tokenizer.from_files(ART_DIR / "tinystories_10k-vocab.json",
                                  ART_DIR / "tinystories_10k-merges.txt",
                                  ["<|endoftext|>"])
    owt_tok = Tokenizer.from_files(ART_DIR / "owt_32k-vocab.json",
                                   ART_DIR / "owt_32k-merges.txt",
                                   ["<|endoftext|>"])

    results = {
        "TinyStories 문서 × TS(10K) 토크나이저": compression_ratio(ts_tok, ts_docs),
        "OWT 문서 × OWT(32K) 토크나이저":        compression_ratio(owt_tok, owt_docs),
        "OWT 문서 × TS(10K) 토크나이저  ← (b)":  compression_ratio(ts_tok, owt_docs),
        "TinyStories 문서 × OWT(32K) 토크나이저": compression_ratio(owt_tok, ts_docs),
    }
    print(f"{'조합':42s}{'bytes/token':>13s}{'MB/s':>9s}")
    for k, v in results.items():
        print(f"{k:42s}{v['bytes_per_token']:>13.3f}{v['throughput_bps']/1e6:>9.2f}")
except Exception as e:
    print("실행 불가:", type(e).__name__, e)
    print("→ Tokenizer 구현 + 두 토크나이저 학습이 모두 끝나야 함")

In [ ]:
# (c) throughput → Pile(825GB) 소요 시간 추정
PILE_BYTES = 825 * 1e9

try:
    bps = results["OWT 문서 × OWT(32K) 토크나이저"]["throughput_bps"]
    sec = PILE_BYTES / bps
    print(f"단일 프로세스 throughput : {bps/1e6:.2f} MB/s")
    print(f"Pile 825GB 예상 소요      : {sec/3600:>10.1f} 시간  ({sec/86400:.1f} 일)")
    for nproc in (4, 8, 16):
        print(f"  {nproc:2d} 프로세스 병렬 가정  : {sec/3600/nproc:>10.1f} 시간")
except (NameError, KeyError):
    print("위 셀을 먼저 실행할 것")

## (d) 데이터셋 전체 인코딩 → `uint16` `.npy`

### 왜 uint16인가
| dtype | 표현 범위 | 바이트/토큰 | 우리 vocab(10K / 32K) |
|---|---|---|---|
| `uint8` | 0 ~ 255 | 1 | ❌ 부족 |
| **`uint16`** | **0 ~ 65,535** | **2** | ✅ 충분 |
| `int32` | ±21억 | 4 | 낭비 (2배 메모리) |

TinyStories 10,000 · OWT 32,000 모두 65,536 미만 → **정보 손실 없이 가장 작은 타입**.
학습 중에는 이 배열을 `np.memmap`으로 열어두고 배치를 잘라 쓰므로,
**파일 크기 = 디스크 I/O 부담 = 학습 속도**에 직결된다. int32면 그냥 두 배 느려진다.

> ⚠️ vocab을 65,536 이상으로 키우는 실험(Phase 6)을 한다면 uint16이 **조용히 wrap-around** 한다.
> `assert max_id < 65536`를 인코딩 스크립트에 넣어둘 것.

### 실행 스크립트

In [ ]:
# scripts/encode_dataset.py 가 준비되어 있는지 확인
enc_path = REPO_ROOT / "scripts" / "encode_dataset.py"
print("존재:", enc_path.exists(), "|", enc_path)
if enc_path.exists():
    import subprocess
    print()
    print(subprocess.run([sys.executable, str(enc_path), "--help"],
                         capture_output=True, text=True).stdout)
else:
    print("→ scripts/encode_dataset.py 를 repo에 배치할 것")

### 실행 (Git Bash) — 4개 파일 전부

```bash
# TinyStories
uv run python scripts/encode_dataset.py \
  --input data/TinyStoriesV2-GPT4-train.txt \
  --vocab artifacts/tinystories_10k-vocab.json \
  --merges artifacts/tinystories_10k-merges.txt \
  --output data/tokenized/tinystories_train.npy

uv run python scripts/encode_dataset.py \
  --input data/TinyStoriesV2-GPT4-valid.txt \
  --vocab artifacts/tinystories_10k-vocab.json \
  --merges artifacts/tinystories_10k-merges.txt \
  --output data/tokenized/tinystories_valid.npy

# OWT
uv run python scripts/encode_dataset.py \
  --input data/owt_train.txt \
  --vocab artifacts/owt_32k-vocab.json \
  --merges artifacts/owt_32k-merges.txt \
  --output data/tokenized/owt_train.npy

uv run python scripts/encode_dataset.py \
  --input data/owt_valid.txt \
  --vocab artifacts/owt_32k-vocab.json \
  --merges artifacts/owt_32k-merges.txt \
  --output data/tokenized/owt_valid.npy
```

**순서 팁**: valid 파일 2개를 먼저 돌려서 파이프라인을 검증하고, train은 그다음.
train은 시간이 오래 걸리므로 진행 로그(`__M tokens`)가 꾸준히 올라가는지 확인.

**성공 판정**: `data/tokenized/` 에 `.npy` 4개, dtype이 `uint16`, 압축률이 (a)에서 잰 값과 비슷.

In [ ]:
# 산출물 검증 — 여기까지 통과하면 Phase 1 완료
import numpy as np

TOK_DIR = DATA_DIR / "tokenized"
for name in ["tinystories_train.npy", "tinystories_valid.npy", "owt_train.npy", "owt_valid.npy"]:
    p = TOK_DIR / name
    if not p.exists():
        print(f"{name:26s} [없음]")
        continue
    arr = np.load(p, mmap_mode="r")          # 학습 때와 동일하게 memmap으로 열기
    print(f"{name:26s} {arr.dtype}  {len(arr):>13,} tokens  {p.stat().st_size/1e9:6.3f} GB  "
          f"min={arr[:1_000_000].min()} max={arr[:1_000_000].max()}")

In [ ]:
# 라운드트립 최종 확인: .npy 앞부분을 디코딩해서 원문과 같은지 눈으로 검증
try:
    arr = np.load(TOK_DIR / "tinystories_valid.npy", mmap_mode="r")
    decoded = ts_tok.decode(arr[:200].tolist())
    print("--- .npy 에서 복원한 텍스트 ---")
    print(decoded)
    print()
    with open(DATA_DIR / "TinyStoriesV2-GPT4-valid.txt", encoding="utf-8") as f:
        original = f.read(len(decoded.encode("utf-8")) + 50)
    print("--- 원본 앞부분 ---")
    print(original[:len(decoded)])
    print()
    print("일치:", original.startswith(decoded[:100]))
except Exception as e:
    print("실행 불가:", type(e).__name__, e)

### 📝 서술형 답안 초안 — `tokenizer_experiments`

**(a)** TinyStories 문서 10개를 TS(10K) 토크나이저로 인코딩하면 압축률 `___` bytes/token,
OWT 문서 10개를 OWT(32K)로 인코딩하면 `___` bytes/token이다.
OWT 쪽이 더 높은 이유는 vocab이 3배 이상 커서 더 긴 단위를 한 토큰에 담기 때문이다
(다만 텍스트 자체의 어휘 다양성이 이를 일부 상쇄한다).

**(b)** OWT 문서를 TinyStories 토크나이저로 인코딩하면 압축률이 `___` → `___` bytes/token으로 **악화**된다.
TS vocab에는 고유명사·전문 용어·URL 조각·비영어 문자열에 해당하는 토큰이 없어,
해당 부분이 거의 **바이트 단위로 쪼개지기** 때문이다. 정성적으로는 짧은 토큰이 길게 이어진 형태가 관찰된다.
같은 문맥 길이(context_length)에 담기는 실제 정보량이 줄어들므로, 모델 성능에 직접 손해다.

**(c)** 단일 프로세스 throughput은 약 `___` MB/s였다. Pile 825GB를 이 속도로 처리하면 약 `___`시간
(≈ `___`일)이 걸린다. 문서 단위로 병렬화가 가능하므로 N 프로세스면 대략 N배 단축된다.

**(d)** 학습에 쓸 토큰 ID의 최댓값이 TinyStories 9,999 / OWT 31,999로 **`uint16`의 상한 65,535보다 작다.**
따라서 uint16이 정보 손실 없이 표현 가능한 가장 작은 타입이고, int32 대비 파일과 메모리를 **절반**으로 줄인다.
학습 중 `np.memmap`으로 읽는 배열이므로 이 차이가 디스크 I/O 처리량에 그대로 반영된다.

---
# 7. Phase 1 완료 체크리스트

실행하며 하나씩 채울 것.

- [ ] 데이터 4개 다운로드 (`TinyStories train/valid`, `owt train/valid`)
- [ ] `cs336_basics/tokenizer/serialization.py` 작성 + 라운드트립 확인
- [ ] `scripts/train_tokenizer.py` 로 TinyStories 10K 학습 (≤30분) → `artifacts/tinystories_10k-*`
- [ ] cProfile 로 병목 확인 → `train_bpe_tinystories (b)` 답안
- [ ] OWT 32K 학습 (smoke → 본 학습) → `artifacts/owt_32k-*`
- [ ] 두 vocab 비교 → `train_bpe_expts_owt (b)` 답안
- [ ] `cs336_basics/tokenizer/tokenizer.py` 구현 (`_bpe`, `encode_iterable`, `decode`)
- [ ] `tests/adapters.py::get_tokenizer` 연결
- [ ] 핸드아웃 예제 `[9, 7, 1, 5, 10, 3]` 통과
- [ ] naive vs rank 방식 대조 통과
- [ ] `uv run pytest tests/test_tokenizer.py -q` 전부 통과 (**15pt**)
- [ ] 압축률/throughput 실험 → `tokenizer_experiments (a)(b)(c)` 답안
- [ ] `.npy` 4개 생성 + dtype uint16 확인 + 라운드트립 검증
- [ ] `writeup/phase1.md` 에 서술형 답안 4개 정리

---

# 8. 다음 단계

Phase 1이 끝나면 `03_transformer_components.ipynb`로 이동.
Linear → Embedding → RMSNorm → SwiGLU → RoPE → softmax → SDPA → MHA → Block → LM 순서로
각각 `pytest -k <name>`을 하나씩 통과시키며 쌓아 올린다.

> **Phase 6 ablation을 미리 대비할 것**: `--norm=none|pre|post`, `--pos_emb=rope|none`,
> `--ffn=swiglu|silu` 플래그로 스위치할 수 있게 03에서부터 구조를 잡아두면
> 나중에 모델 코드를 다시 뜯어고칠 일이 없다.